# Setting up the stage - Markov Decision Processes

Let's introduce our guiding example, a *Vehicle-to-Grid* (V2G) management problem where a power supplier needs to charge electric vehicles (EVs) arriving randomly along the day, while optimizing its operational costs.

In [1]:
from ev2gym.models.ev2gym_env import EV2Gym
from ev2gym.baselines.mpc.V2GProfitMax import V2GProfitMaxOracle
from ev2gym.baselines.heuristics import ChargeAsFastAsPossible

config_file = "ev2gym/example_config_files/V2GProfitPlusLoads.yaml"

# Initialize the environment
env = EV2Gym(config_file=config_file,
              save_replay=True,
              save_plots=True)

/home/emmanuel/anaconda3/lib/python3.13/site-packages/ev2gym/utilities/loaders.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Creating directory: ./results/sim_2026_03_17_206618


Let's take a peek into the config file.

In [2]:
from pygments import highlight
from pygments.formatters import Terminal256Formatter
from pygments.lexers import YamlLexer

config_file_contents = open(config_file).read()
print(highlight(config_file_contents, YamlLexer(), Terminal256Formatter()))

# This yml file is used to configure the evsim simulation

##############################################################################
# Simulation Parameters
##############################################################################
timescale: 15 # in minutes per step
simulation_length: 112 #90 # in steps per simulation

##############################################################################
# Date and Time
##############################################################################
# Year, month, 
year: 2022 # 2015-2023
month: 1 # 1-12
day: 17 # 1-31
# Whether to get a random date every time the environment is reset
random_day: True # True or False
random_hour: False # True or False

# Simulation Starting Time
# Hour and minute do not change after the environment has been reset
hour: 5 # Simulation starting hour (24 hour format)
minute: 0 # Simulation starting minute (0-59)

# Simulate weekdays, weekends, or both
simulation_days: weekdays # weekdays, weekends, or both

There are lots of lines in this configuration file. Let's summarize.  
We are simulating 112 time steps of 15-minutes length (that's 28 hours in total), on a random day in January 2022, starting at 5am. The scenario follows a "weekdays" profile and the vehicles arrive and leave as if the location is a "workplace".  
The distribution network we operate has a single transformer, 25 charging stations, each with a single EV charging port.  
The vehicles are heterogenous.

In [3]:
print(len(env.transformers))
print(len(env.charging_stations))
print(env.number_of_ports_per_cs)

1
25
1


Let's take a look at the initial state of our system.

In [4]:
state, _ = env.reset()

print("Connected EVs at each charging station")
for cs in env.charging_stations:
    print(cs.evs_connected)

print("\nState variables")
print(state)
print(len(state))
# "state" holds the following variables:
# current_step / simulation_length
# power_setpoint[current_step]
# power_usage[current_step-1]
# for each EV, [a,b,c] where 
#   a=b=c=0 if vehicle not connected, 
#   a=1 if vehicle fully charged, otherwise 0.5
#   b=total energy exchanged with vehicle
#   c=stay duration of vehicle
# So, overall, that is 3*(1+nb_vehicles) variables.

import numpy as np
print("\nA prettier display of state variables")
def reshape_state(env, s):
    return np.reshape(s, (1+env.cs*env.number_of_ports_per_cs,3))

print(reshape_state(env, state))

Connected EVs at each charging station
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]

State variables
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0.]
78

A prettier display of state variables
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


The decision variables for our problem are the current intensity, for each charging port, at every time step.

In [5]:
print(env.action_space) # one action variable per charging station (power output, between -1 and 1)

Box(-1.0, 1.0, (25,), float64)


Let's run a simulation.  
For this, we need a function that provides the decided current intensity at each time step. That's what we will call an *agent*.

In [6]:
#agent = V2GProfitMaxOracle(env)#,verbose=True) # optimal solution
#        or 
agent = ChargeAsFastAsPossible() # heuristic
for t in range(env.simulation_length):
    actions = agent.get_action(env) # get action from the agent
    new_state, _, done, truncated, stats = env.step(actions)  # takes action

Saving replay file at ./replay/replay_sim_2026_03_17_206618.pkl
Plotting simulation data at ./results/sim_2026_03_17_206618/


/home/emmanuel/anaconda3/lib/python3.13/site-packages/ev2gym/visuals/plots.py:651: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


In [7]:
stats

{'total_ev_served': np.int64(17),
 'total_profits': np.float64(-17.088152273813293),
 'total_energy_charged': np.float64(223.78068839591322),
 'total_energy_discharged': np.int64(0),
 'average_user_satisfaction': np.float64(1.0),
 'power_tracker_violation': np.float64(895.1227535836529),
 'tracking_error': np.float64(27468.081092942102),
 'energy_tracking_error': np.float64(223.78068839591322),
 'energy_user_satisfaction': np.float64(100.0),
 'std_energy_user_satisfaction': np.float64(0.0),
 'min_energy_user_satisfaction': np.float64(100.0),
 'total_steps_min_emergency_battery_capacity_violation': 0,
 'total_transformer_overload': np.float64(0.7047638326794328),
 'battery_degradation': np.float64(0.0006818329993215145),
 'battery_degradation_calendar': np.float64(0.0004360364436469144),
 'battery_degradation_cycling': np.float64(0.00024579655567460003),
 'total_reward': np.float64(-27468.081092942102),
 'saved_grid_energy': 0,
 'voltage_violation': 0,
 'voltage_violation_counter': 0,
 

Let's re-run that step by step, to see how the state changes. Run the second cell below repeatedly.

In [8]:
state, _ = env.reset()

In [48]:
action = agent.get_action(env)
new_state, _, done, truncated, stats = env.step(action)  # takes action
print(env.current_step)
print(action)
print(reshape_state(env, new_state))
#print(reward)

40
[0. 0. 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 0. 1. 1. 0. 1. 1. 1. 0. 1. 1. 1. 1.
 0.]
[[ 0.35714286  0.          9.9628512 ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 1.         26.94979208 14.        ]
 [ 1.         10.47771337 21.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 1.         10.8532484  30.        ]
 [ 0.          0.          0.        ]
 [ 1.         15.87554007 27.        ]
 [ 1.         18.31352185 20.        ]
 [ 1.         17.4232979  24.        ]
 [ 1.         25.77803927 26.        ]
 [ 0.          0.          0.        ]
 [ 1.          7.95829845 21.        ]
 [ 0.5         4.95        2.        ]
 [ 0.          0.          0.        ]
 [ 1.         25.56151486 21.        ]
 [ 1.         11.74460283 25.        ]
 [ 1.         22.69006244 25.        ]
 [ 0.          0.          0.        ]
 [ 1.         20.35773321 28.        ]
 [ 1.         23.48627614 19.        ]
 [ 1.          4.98048

In [49]:
env.charging_stations[2].evs_connected[0].time_of_departure

53

In [ ]:
help(env)

In [ ]:
print(env.current_step)
print(env.power_setpoints[env.current_step])
print(env.power_setpoints)
print(env.current_power_usage)

In [ ]:
print(len(env.charging_stations))

In [61]:
from ev2gym.models.ev2gym_env import EV2Gym
from ev2gym.baselines.mpc.V2GProfitMax import V2GProfitMaxOracle
from ev2gym.baselines.heuristics import ChargeAsFastAsPossible
from ev2gym.rl_agent.state import V2G_profit_max

#config_file = "ev2gym/example_config_files/V2GProfitPlusLoads.yaml"
config_file = "custom.yaml"

# Initialize the environment
env = EV2Gym(config_file=config_file,
              save_replay=True,
              save_plots=True,
              state_function=V2G_profit_max)

Creating directory: ./results/sim_2026_03_17_920596


In [62]:
state, _ = env.reset()
print("Connected EVs at each charging station")
for cs in env.charging_stations:
    print(cs.evs_connected)

print("\nState variables")
print(state)
print(len(state))

Connected EVs at each charging station
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]

State variables
[0.      0.      0.1087  0.1087  0.1087  0.1087  0.1118  0.1118  0.1118
 0.1118  0.0949  0.0949  0.0949  0.0949  0.08438 0.08438 0.08438 0.08438
 0.08205 0.08205 0.08205 0.08205 0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.     ]
42


In [63]:
import time

agent = ChargeAsFastAsPossible() # heuristic
start = time.time()
for ep in range(1):
    print("episode", ep)
    env.reset()
    for t in range(env.simulation_length):
        actions = agent.get_action(env) # get action from the agent
        new_state, _, done, truncated, stats = env.step(actions)  # takes action

end = time.time()
print(end - start)

episode 0
Saving replay file at ./replay/replay_sim_2026_03_17_920596.pkl
Plotting simulation data at ./results/sim_2026_03_17_920596/
3.3904759883880615
